# 1. 사용한 모델과 데이터셋 

### 1.1 프로젝트 목표 

스킨케어 제품은 단순히 평점이 높은 것보다, 자신의 피부 고민에 맞는지가 더 중요하다.

이번 프로젝트를 통해서 스킨 케어 리뷰 한 문장에서 사용자의 피부 고민과 관련된 여러 만족 요소를 동시에 찾아내는 멀티 라벨 분류 모델을 만드는 것이다.

이 모델에서는 아래의 다섯 가지 만족도를 예측한다
- 보습,수분감 만족
- 흡수력
- 발림성
- 저자극 관련 긍정 반응
- 윤기, 피부톤 개선 

이를 통해 사용자는 단순한 평균 별점 대신, 자신이 중요하게 생각하는 기준에 따라 리뷰를 해석할 수 있다. 



### 1.1 KLUE/RoBERTa-base

이번 프로젝트에선느 Hugging Face의 klue/roberta-base 모델을 사용했다


### 1.2 데이터셋 

- 데이터 출처 : AI Hub
- 데이터셋명 : 속성 기반 감정분석 데이터
- 사용 범위 : 쇼핑몰-화장품-스킨케어 리뷰



# 2. Tokenizer를 이용한 데이터 전처리

In [1]:
# 기본 모듈
from pathlib import Path
import json
import random
import time
import datetime

# 데이터 처리
import pandas as pd
import numpy as np

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch
import torch
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

# Hugging Face
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

# 평가
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    hamming_loss,
    classification_report,
    roc_auc_score
)

/Users/99yoo/Desktop/kdt/논문/트랜스포머/Transformer_Assignment/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("DEVICE:", DEVICE)

DEVICE: mps


In [3]:
SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"

if not DATA_DIR.exists():
    DATA_DIR.mkdir()
else:
    print("이미 저장소가 존재합니다", DATA_DIR)

이미 저장소가 존재합니다 /Users/99yoo/Desktop/kdt/논문/트랜스포머/Transformer_Assignment/data


In [5]:
TRAIN_LABEL_DIR= next(
  path for path in (DATA_DIR/"Training"/"02.라벨링데이터").iterdir()
  if path.is_dir()
)

print(TRAIN_LABEL_DIR)

/Users/99yoo/Desktop/kdt/논문/트랜스포머/Transformer_Assignment/data/Training/02.라벨링데이터/TL_쇼핑몰_02.화장품_2-1.스킨케어


In [6]:
VALID_LABEL_DIR = next(
    path for path in (DATA_DIR / "Validation" / "02.라벨링데이터").iterdir()
    if path.is_dir()

)

print(VALID_LABEL_DIR)

/Users/99yoo/Desktop/kdt/논문/트랜스포머/Transformer_Assignment/data/Validation/02.라벨링데이터/VL_쇼핑몰_02.화장품_2-1.스킨케어


학습 데이터들(json) 다 합치기 -> 다섯가지 만족도 요소를 인덱스로 -> train_df 로 만들기 

In [7]:
train_data = {
    "items": []
}

for json_file in sorted(TRAIN_LABEL_DIR.glob("*.json")):
    with json_file.open("r", encoding="utf-8") as f:
        json_data = json.load(f)

    # JSON 파일의 바깥 구조가 [...]인 경우
    if isinstance(json_data, list):
        train_data["items"].extend(json_data)

    # JSON 파일의 바깥 구조가 {"item": [...]}인 경우
    else:
        train_data["items"].extend(json_data["item"])

print("전체 학습 리뷰 수:", len(train_data["items"]))

전체 학습 리뷰 수: 12000


In [8]:
first_review = train_data["items"][100]
print("리뷰:", first_review["RawText"])

print("Aspects:", first_review["Aspects"])

리뷰: 민감한피부인데 저한테 잘맞아요 피부를 진정시켜주네요 백신맞고 아토피 올라와서 당분간 잘사용할 예정이에요 시어버터향는 개인적으로 좀 안좋아하는데 없는걸루 신제품이 개발됬음 좋겠네요
Aspects: [{'Aspect': '기능/효과', 'SentimentText': '민감한피부인데 저한테 잘맞아요', 'SentimentWord': '3', 'SentimentPolarity': '1'}, {'Aspect': '기능/효과', 'SentimentText': '피부를 진정시켜주네요 ', 'SentimentWord': '2', 'SentimentPolarity': '1'}, {'Aspect': '향', 'SentimentText': '시어버터향는 개인적으로 좀 안좋아하는데 없는걸루 신제품이 개발됬음 좋겠네요', 'SentimentWord': '8', 'SentimentPolarity': '-1'}]


In [9]:
LABEL_TO_ASPECT = {
    "보습_수분감_만족": "보습력/수분감",
    "흡수력_만족": "흡수력",
    "발림성_만족": "발림성",
    "저자극_관련_긍정": "자극성",
    "윤기_피부톤_개선": "윤기/피부(톤)",
}

CLASS_NAMES = list(LABEL_TO_ASPECT.keys())

ASPECT_TO_ID = {
    aspect_name: label_id
    for label_id, aspect_name in enumerate(LABEL_TO_ASPECT.values())
}

print("클래스:", CLASS_NAMES)
print("속성 → 라벨 번호:", ASPECT_TO_ID)

클래스: ['보습_수분감_만족', '흡수력_만족', '발림성_만족', '저자극_관련_긍정', '윤기_피부톤_개선']
속성 → 라벨 번호: {'보습력/수분감': 0, '흡수력': 1, '발림성': 2, '자극성': 3, '윤기/피부(톤)': 4}


In [10]:
from collections import Counter

In [11]:
aspect_counts = Counter()

for review in train_data["items"]:
    for ann in review["Aspects"]:
        aspect_name = ann["Aspect"]

        if aspect_name in ASPECT_TO_ID:
            aspect_counts[aspect_name] += 1

for aspect_name, label_id in ASPECT_TO_ID.items():
    print(
        f"{label_id}: {aspect_name} "
        f"> {aspect_counts[aspect_name]:,}개 annotation"
    )

0: 보습력/수분감 > 7,039개 annotation
1: 흡수력 > 3,251개 annotation
2: 발림성 > 1,745개 annotation
3: 자극성 > 1,290개 annotation
4: 윤기/피부(톤) > 2,376개 annotation


In [12]:
rows = []

for review in train_data["items"]:
    text = review["RawText"]

    # [0, 0, 0, 0, 0]으로 시작
    labels = [0] * len(CLASS_NAMES)

    # 이 리뷰에 붙은 모든 속성 annotation 확인
    for ann in review["Aspects"]:
        aspect_name = ann["Aspect"]
        polarity = str(ann["SentimentPolarity"])

        # 우리가 선택한 속성이면서 긍정(1)일 때만 1 처리
        if aspect_name in ASPECT_TO_ID and polarity == "1":
            label_id = ASPECT_TO_ID[aspect_name]
            labels[label_id] = 1

    # DataFrame 한 행으로 만들기
    row = {
        "text": text
    }

    for label_id, class_name in enumerate(CLASS_NAMES):
        row[class_name] = labels[label_id]

    rows.append(row)

In [13]:
train_df = pd.DataFrame(rows)

print("학습 데이터 크기:", train_df.shape)


학습 데이터 크기: (12000, 6)


In [14]:
train_df.tail(10)

,text,보습_수분감_만족,흡수력_만족,발림성_만족,저자극_관련_긍정,윤기_피부톤_개선
11990,OOO크림을 바르면서 가려움이 없어졌다고 합니다. 촉촉한 보습감이 좋네요.,1,0,0,0,0
11991,순하고 부드러워서 계속 쓰고 있습니다. 홈쇼핑에서 대량으로 사다가 처음으로 소량 구...,0,0,0,1,1
11992,향이 병풀향이라는데 독특한데 저랑 안 맞네요 크림 자체는 촉촉합니다 아직 한번만 써...,1,0,0,0,0
11993,부드럽게 발리고 흡수도 잘되고 건조함이 덜해 좋네요,1,1,1,0,0
11994,친구 추천으로 알아보다가 착한가격에 잘 샀습니다. 사용기간도 2년후까지 여유있게 넉...,0,0,0,0,0
11995,향은 좀 약냄새가 나는듯 하지만 발랐을때 정말 잘 스며들고 트러블도 생기지 않습니다...,0,1,0,1,0
11996,촉촉하고 쫀쫀하게 코팅되는 느낌입니다. OOO 특유의 향이 있는데 저는 괜찮네요. ...,1,0,0,0,0
11997,얼굴에 붉은기가 올라오고 따가웠는데 세안 후 스킨 바른 다음 OOO만 발랐는데도 붉...,0,0,0,0,0
11998,써보니 순하고 괜찮은 것 같아서 재구매했습니다. 낮에 바를 때 번들거리는 게 없어 ...,0,1,0,1,0
11999,저렴하게 구매했네요. 발림도 좋고 피부가 쫀득쫀득해지는거 같습니다.,0,0,1,0,0


In [15]:
print(train_df.columns.tolist())


['text', '보습_수분감_만족', '흡수력_만족', '발림성_만족', '저자극_관련_긍정', '윤기_피부톤_개선']


validation json도 위의 학습 데이터와 같은 방식으로 valid_df 만들기 

In [16]:
valid_data = {
  "items":[]
}

for json_file in sorted(VALID_LABEL_DIR.glob("*.json")):
  with json_file.open("r", encoding="utf-8") as f:
    json_data = json.load(f)

  if isinstance(json_data,list):
    valid_data['items'].extend(json_data)
  else:
    valid_data['items'].extend(json_data["item"])

print('전체 검증 데이터 리뷰 수:', len(valid_data['items']))

전체 검증 데이터 리뷰 수: 1500


In [17]:
valid_sample = valid_data['items'][10]
print(valid_sample)

{'Index': '582270', 'RawText': 'OOO 오일은 처음인데 제형이 꾸덕하지 않고  순해서 좋은 것 같습니다. 향은 좀 강하네요. 겨울이나 건조한 날씨에 사용하기 좋은 것 같습니다. 건성이신 분들에게 추천해요. 착한 가격은 아니에요.', 'Source': '쇼핑몰', 'Domain': '화장품', 'MainCategory': '스킨케어', 'ProductName': 'OO 순** 클렌징오일 200ml 1개', 'ReviewScore': '5', 'Syllable': '109', 'Word': '26', 'RDate': '20220126', 'GeneralPolarity': '1', 'Aspects': [{'Aspect': '제형', 'SentimentText': '제형이 꾸덕하지 않고 ', 'SentimentWord': '3', 'SentimentPolarity': '1'}, {'Aspect': '자극성', 'SentimentText': '순해서 ', 'SentimentWord': '1', 'SentimentPolarity': '1'}, {'Aspect': '향', 'SentimentText': '향은 좀 강하네요', 'SentimentWord': '3', 'SentimentPolarity': '-1'}, {'Aspect': '기능/효과', 'SentimentText': '겨울이나 건조한 날씨에 사용하기 좋은 것 같습니다.', 'SentimentWord': '7', 'SentimentPolarity': '1'}, {'Aspect': '피부타입', 'SentimentText': '건성이신 분들에게 추천해요.', 'SentimentWord': '3', 'SentimentPolarity': '1'}, {'Aspect': '가격', 'SentimentText': '착한 가격은 아니에요.', 'SentimentWord': '3', 'SentimentPolarity': '-1'}]}


In [18]:
valid_rows =[]
for review in valid_data['items']:
  text = review['RawText']

  labels = [0] * len(CLASS_NAMES)

  for ann in review['Aspects']:
    aspect_name= ann['Aspect']
    polarity= str(ann['SentimentPolarity'])

    if aspect_name in ASPECT_TO_ID and polarity == "1":
      label_id = ASPECT_TO_ID[aspect_name]
      labels[label_id] = 1

  row = {
    "text": text,
  }
  for label_id, class_name in enumerate(CLASS_NAMES):
    row[class_name] = labels[label_id]

  valid_rows.append(row)

In [19]:
valid_df = pd.DataFrame(valid_rows)
print("검증 데이터 크기",valid_df.shape)

검증 데이터 크기 (1500, 6)


현재 데이터가 train과 validation 밖에 없어서 validation_df를 test_df로 하고 

기존 train_df를 9대1의 비율로 train /validation으로 하겠다

In [21]:
test_df = valid_df.copy() 

In [22]:
train_df , valid_df = train_test_split(
  train_df,
  test_size= 0.1,
  random_state= SEED
)

## 모델이랑 Tokenizer 불러오기

In [23]:
MODEL_NAME='klue/roberta-base'
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer:", type(tokenizer).__name__)
print("CLS:", tokenizer.cls_token_id, tokenizer.cls_token)
print("SEP:", tokenizer.sep_token_id, tokenizer.sep_token)
print("PAD:", tokenizer.pad_token_id, tokenizer.pad_token)
print("UNK:", tokenizer.unk_token_id, tokenizer.unk_token)
print("MASK:", tokenizer.mask_token_id, tokenizer.mask_token)

Tokenizer: BertTokenizer
CLS: 0 [CLS]
SEP: 2 [SEP]
PAD: 1 [PAD]
UNK: 3 [UNK]
MASK: 4 [MASK]


In [24]:
# 학습 데이터에서 샘플 데이터 하나를 뽑아서 토크나이징 해보기
sample_text = train_df.iloc[0]['text']

tokenized_sample = tokenizer(
    sample_text,
    add_special_tokens=True,
    padding='max_length',
    max_length=MAX_LEN,

    truncation=True,
    return_tensors='pt'
)
print("토크나이징되기전 텍스트:", sample_text)
print("토크나이징된 결과:", tokenized_sample)

토크나이징되기전 텍스트: 바르면 촉촉하고 얼굴이 생기있어 보여 수시로 덧바르게 되네요. 이번에도 잘 쓰게씁니다~
토크나이징된 결과: {'input_ids': tensor([[    0,  5957,  2460,  8974, 19521,  3977,  2052,  4911,  2689,  2051,
          3897,  8895,   838, 23746,  2318,   859,  2203,  2182,    18,  3686,
          6509,  1521,  1363,  2318,  3465,  3606,    97,     2,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,

In [25]:

def encode_texts(texts, labels, tokenizer, max_len=MAX_LEN):
    encoded = tokenizer(
        ["" if x is None else str(x) for x in texts],
        add_special_tokens=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
        padding="max_length",
    )

    return (
        encoded["input_ids"],
        encoded["attention_mask"],
        labels.to(torch.float32),
    )

In [26]:
# 라벨 DataFrame → PyTorch Tensor
train_label_tensor = torch.tensor(
    train_df[CLASS_NAMES].values,
    dtype=torch.float32,
)

validation_label_tensor = torch.tensor(
    valid_df[CLASS_NAMES].values,
    dtype=torch.float32,
)

test_label_tensor = torch.tensor(
    test_df[CLASS_NAMES].values,
    dtype=torch.float32,
)

In [27]:
train_inputs, train_mask, train_labels = encode_texts(
    train_df["text"],
    train_label_tensor,
    tokenizer,
)

validation_inputs, validation_mask, validation_labels = encode_texts(
    valid_df["text"],
    validation_label_tensor,
    tokenizer,
)

test_inputs, test_mask, test_labels = encode_texts(
    test_df["text"],
    test_label_tensor,
    tokenizer,
)

In [28]:
print("train input_ids:", train_inputs.shape)
print("train attention_mask:", train_mask.shape)
print("train labels:", train_labels.shape)

train input_ids: torch.Size([10800, 128])
train attention_mask: torch.Size([10800, 128])
train labels: torch.Size([10800, 5])


데이터 로더 만들기

In [29]:
BATCH_SIZE= 32

train_data = TensorDataset(train_inputs, train_mask, train_labels)
validation_data = TensorDataset(validation_inputs, validation_mask, validation_labels)
test_data = TensorDataset(test_inputs, test_mask, test_labels)

In [30]:
# 배치 사이즈만큼 데이터로더가 꺼내서 모델에 전달시키기

train_dataloader = DataLoader(
  train_data,
  sampler=RandomSampler(train_data),
  batch_size=BATCH_SIZE
)

validation_dataloader = DataLoader(
  validation_data,
  sampler=RandomSampler(validation_data),
  batch_size=BATCH_SIZE
)

test_dataloader = DataLoader(
  test_data,
  sampler=RandomSampler(test_data),
  batch_size=BATCH_SIZE
)

In [31]:
batch = next(iter(train_dataloader))
print("input_ids:", batch[0].shape)
print("attention_mask:", batch[1].shape)
print("labels:", batch[2].shape)

input_ids: torch.Size([32, 128])
attention_mask: torch.Size([32, 128])
labels: torch.Size([32, 5])


# 3. 사전 학습 모델 Fine-Tuning


In [32]:
print(DEVICE)

mps


In [33]:
num_labels = len(CLASS_NAMES)

label2id = {
    label_name: label_id
    for label_id, label_name in enumerate(CLASS_NAMES)
}

id2label = {
    label_id: label_name
    for label_id, label_name in enumerate(CLASS_NAMES)
}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    problem_type="multi_label_classification",
    label2id=label2id,
    id2label=id2label,
)

model.to(DEVICE)

print(type(model).__name__)
print("라벨 수:", model.config.num_labels)
print("라벨 이름:", model.config.id2label)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 11081.76it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification
라벨 수: 5
라벨 이름: {0: '보습_수분감_만족', 1: '흡수력_만족', 2: '발림성_만족', 3: '저자극_관련_긍정', 4: '윤기_피부톤_개선'}


In [34]:
model.to(DEVICE)

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
  

In [35]:
print(type(model).__name__)
print(model.config.num_labels)

RobertaForSequenceClassification
5


In [36]:
EPOCHS = 5
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
BATCH_SIZE = 16

optimizer = AdamW(
  model.parameters(),
  lr=LEARNING_RATE,
  weight_decay=WEIGHT_DECAY
)

total_steps = len(train_dataloader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
  optimizer,
  num_warmup_steps=int(total_steps * WARMUP_RATIO),
  num_training_steps=total_steps
)

print('전체 학습 step:', total_steps)
print('워밍업 step:', int(total_steps * WARMUP_RATIO))



전체 학습 step: 1690
워밍업 step: 101


In [37]:
def multi_label_metrics(logits, labels, threshold =0.5):
  probs = torch.sigmoid(
    torch.as_tensor(logits)
).cpu().numpy()
  y_true= torch.as_tensor(labels).cpu().numpy()

  # 확률이 0.5이상이면 해당 라벨을 예측한 것으로 처리하기
  y_pred = (probs >= threshold).astype(int)

  metrics ={
    "subset_accuracy": accuracy_score(y_true, y_pred),
    "f1_micro": f1_score(y_true,y_pred, average='micro', zero_division=0),
    "f1_macro": f1_score(y_true,y_pred, average='macro', zero_division=0),
    "f1_weighted": f1_score(y_true,y_pred, average='weighted', zero_division=0),
    'hamming_loss': hamming_loss(y_true, y_pred)

  }

  try:
    metrics["roc_auc_micro"] = roc_auc_score(
    y_true,
    probs,
    average="micro",
)

  except ValueError:
    metrics['roc_auc_micro'] = float('nan')

  return metrics

In [38]:
def evaluate(model, dataloader, device):
    model.eval()  # 평가 모드: dropout 등을 비활성화

    all_logits = []
    all_labels = []

    for batch in tqdm(dataloader, leave=False):
        # TensorDataset에서 넣은 순서와 동일
        input_ids, attention_mask, labels = [
            x.to(device) for x in batch
        ]

        # 평가에서는 gradient 계산 불필요
        with torch.inference_mode():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

        # MPS/GPU Tensor를 CPU로 옮겨서 나중에 지표 계산
        all_logits.append(outputs.logits.cpu())
        all_labels.append(labels.cpu())

    # 모든 배치 결과를 하나로 합침
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()

    # 앞에서 만든 멀티 라벨 평가 함수 호출
    return multi_label_metrics(logits, labels)

# 4. 학습 과정 및 성능 평가

In [39]:
MODEL_DIR = Path("model")
MODEL_DIR.mkdir(exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "best_klue_roberta_skincare.pt"

best_f1 = -1.0
patience = 2
no_improve_count = 0

history = []

In [40]:
def format_time(elapsed):
    return str(datetime.timedelta(seconds=int(round(elapsed))))

In [41]:
for epoch in range(EPOCHS):
    print(f"\n======== Epoch {epoch + 1} / {EPOCHS} ========")

    start = time.time()
    model.train()
    total_loss = 0.0

    for batch in tqdm(train_dataloader):
        input_ids, attention_mask, labels = [
            x.to(DEVICE) for x in batch
        ]

        # 이전 배치에서 계산된 gradient 제거
        optimizer.zero_grad(set_to_none=True)

        # labels를 같이 넣으면 멀티 라벨용 loss가 자동 계산됨
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs.loss
        total_loss += loss.item()

        # 역전파
        loss.backward()

        # gradient가 과도하게 커지는 것을 방지
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        # 모델 가중치와 학습률 업데이트
        optimizer.step()
        scheduler.step()

    avg_loss = total_loss / len(train_dataloader)

    print(f"Train Loss: {avg_loss:.4f}")
    print("Time:", format_time(time.time() - start))

    # validation 데이터로 성능 평가
    valid_metrics = evaluate(
        model,
        validation_dataloader,
        DEVICE,
    )

    print(f"Validation Macro F1: {valid_metrics['f1_macro']:.4f}")
    print(f"Validation Micro F1: {valid_metrics['f1_micro']:.4f}")
    print(f"Hamming Loss: {valid_metrics['hamming_loss']:.4f}")

    # 학습 과정 저장: 나중에 그래프로 활용
    history.append({
        "epoch": epoch + 1,
        "train_loss": avg_loss,
        **valid_metrics,
    })

    # macro F1이 최고일 때만 모델 저장
    if valid_metrics["f1_macro"] > best_f1:
        best_f1 = valid_metrics["f1_macro"]
        no_improve_count = 0

        torch.save(model.state_dict(), BEST_MODEL_PATH)

        print(f"최고 모델 저장 완료 (Macro F1: {best_f1:.4f})")

    else:
        no_improve_count += 1
        print(f"성능 개선 없음: {no_improve_count}/{patience}")

        # 2 epoch 연속 성능이 개선되지 않으면 종료
        if no_improve_count >= patience:
            print("Early stopping")
            break


======== Epoch 1 / 5 ========


100%|██████████| 338/338 [05:41<00:00,  1.01s/it]


Train Loss: 0.2465
Time: 0:05:41


Validation Macro F1: 0.9350
Validation Micro F1: 0.9423
Hamming Loss: 0.0197
최고 모델 저장 완료 (Macro F1: 0.9350)

======== Epoch 2 / 5 ========


100%|██████████| 338/338 [05:39<00:00,  1.00s/it]


...

Train Loss: 0.0303
Time: 0:05:36


Validation Macro F1: 0.9502
Validation Micro F1: 0.9562
Hamming Loss: 0.0153
최고 모델 저장 완료 (Macro F1: 0.9502)

======== Epoch 5 / 5 ========


100%|██████████| 338/338 [05:37<00:00,  1.00it/s]


Train Loss: 0.0232
Time: 0:05:37


Validation Macro F1: 0.9566
Validation Micro F1: 0.9618
Hamming Loss: 0.0133
최고 모델 저장 완료 (Macro F1: 0.9566)


loaded_model로 최고 모델 가중치를 불러오기

In [42]:
loaded_model = AutoModelForSequenceClassification.from_pretrained(
  MODEL_NAME,
  num_labels=num_labels,
  problem_type="multi_label_classification",
  label2id=label2id,
  id2label=id2label
)

state_dict= torch.load(
  BEST_MODEL_PATH,
  map_location ="cpu"
)

loaded_model.load_state_dict(state_dict)
loaded_model.to(DEVICE)

loaded_model.eval()
print("모델 로드 완료")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 9145.30it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


모델 로드 완료


In [43]:
test_metrics = evaluate(loaded_model, test_dataloader, DEVICE)


In [44]:
from transformers import pipeline
pipe = pipeline(
  task = "text-classification",
  model = loaded_model,
  tokenizer = tokenizer,
  device = DEVICE,
  truncation=True,
  max_length = MAX_LEN,
  function_to_apply="sigmoid",
  top_k= None
)

In [45]:
print(pipe('너무 건조해'))

[[{'label': '보습_수분감_만족', 'score': 0.16594764590263367}, {'label': '윤기_피부톤_개선', 'score': 0.061258260160684586}, {'label': '발림성_만족', 'score': 0.0019494189182296395}, {'label': '저자극_관련_긍정', 'score': 0.0018390767509117723}, {'label': '흡수력_만족', 'score': 0.0011799543863162398}]]


# 5. 새로운 문장을 직접 입력하여 예측

In [46]:
def predict_skincare_labels(text, threshold=0.5):
  results = pipe(text)

  if results and isinstance(results[0], list):
    results = results[0]

  selected = []
  for item in results:
    if item['score'] >= threshold:
      selected.append({
        'label': item['label'],
        'score': round(float(item['score']), 3)
      })

  return selected

In [47]:
new_review = "화이트닝이 잘 되는 지 잘 모르겠어요.. 과대광고인듯합니다.."
print(predict_skincare_labels(new_review))

[]


In [48]:

review = "촉촉하고 발림성도 좋지만 흡수되는 데 시간이 조금 걸려요."

print(predict_skincare_labels(review))

[{'label': '발림성_만족', 'score': 0.99}, {'label': '보습_수분감_만족', 'score': 0.979}]


# 6. 모델의 한계 및 개선 방법

### 1. 제한된 속성 라벨만 사용
AI Hub 데이터에는 기능/효과, 가격, 향, 용량, 피부 타입 등 다양한 속성이 존재한다. 하지만 이번 프로젝트에서는 사용자가 스킨케어 리뷰에서 자주 확인할 만한 `보습/수분감`, `흡수력`, `발림성`, `자극성`, `윤기/피부톤` 5개 속성만 선택했다.

따라서 미백, 여드름 완화, 가격, 향처럼 선택하지 않은 속성이 포함된 문장을 입력하면 모델이 관련 라벨을 예측하지 못하고 빈 리스트(`[]`)를 반환할 수 있다. 이는 문장에 의견이 없다는 뜻이 아니라, **현재 모델이 분류하도록 설정된 5개 속성에 해당하는 긍정 신호를 찾지 못했다는 뜻**이다.

**개선 방법:** 데이터에 존재하는 다른 속성도 추가하거나, 프로젝트 목적에 맞게 `기능/효과`, `가격`, `향`, `피부 타입` 등의 라벨을 확장할 수 있다.

### 2. 긍정 의견 중심의 라벨 구성
현재 라벨은 `보습력/수분감 만족`, `발림성 만족`처럼 각 속성에 대한 긍정 의견을 중심으로 구성했다. 그래서 “흡수되는 데 시간이 오래 걸린다”와 같은 부정적인 의견은 `흡수력 만족` 라벨로 분류되지 않지만, 별도의 `흡수력 불만` 라벨도 없어 구체적인 불만으로 표현할 수 없다.

**개선 방법:** 각 속성을 긍정과 부정으로 나누어 `흡수력_만족 / 흡수력_불만`, `자극성_저자극 / 자극성_자극적`처럼 구성하면 리뷰의 방향까지 더 자세히 분석할 수 있다.

### 3. 제한된 학습 횟수와 하이퍼파라미터 탐색
이번 프로젝트는 시간 제약을 고려하여 최대 5 epoch만 학습했고, 하나의 학습률과 배치 크기 설정을 중심으로 실험했다. 따라서 현재 설정이 이 데이터에 가장 적합한 설정이라고 단정하기는 어렵다.

**개선 방법:** 최대 epoch를 10 정도로 늘린 뒤, validation 성능을 기준으로 Early Stopping을 적용하는 방법을 사용할 수 있다. 또한 학습률, 배치 크기, 최대 토큰 길이 등을 여러 조합으로 비교하여 Macro F1 점수가 가장 높은 설정을 선택할 수 있다. 단순히 epoch만 늘리면 과적합이 발생할 수 있으므로 validation 성능을 함께 확인해야 한다.

### 4. 라벨별 데이터 수 불균형
보습력이나 흡수력처럼 많이 등장하는 속성도 있지만, 일부 속성은 상대적으로 적은 수의 학습 데이터를 가진다. 이 경우 모델이 데이터가 많은 라벨을 더 잘 예측하고, 적은 라벨은 놓칠 가능성이 있다.

**개선 방법:** 라벨별 F1-score를 따로 확인하고, 데이터가 적은 라벨을 추가 확보하거나 class weight, focal loss, 라벨별 threshold 조정 등의 방법을 적용할 수 있다.

### 5. 완전한 개인화 추천 모델은 아님
이 프로젝트는 리뷰 문장에서 사용자의 관심 속성과 관련된 내용을 찾아주는 모델이다. 하지만 사용자의 실제 피부 타입, 알레르기 여부, 선호 성분, 제품 성분 정보까지 반영하지는 않는다. 따라서 “민감성 피부인 특정 사용자에게 가장 적합한 제품을 추천하는 시스템”이라고 보기는 어렵다.

**개선 방법:** 사용자 피부 타입과 고민 정보를 추가로 입력받고, 제품의 성분·가격·피부 타입 정보를 함께 활용한다면 개인화된 스킨케어 추천 시스템으로 확장할 수 있다.